In [ ]:
# Setup: load env, create v2 client
import os
from pathlib import Path
from dotenv import load_dotenv
from md_python import MDClient, Upload, ExperimentDesign, SampleMetadata

load_dotenv()
api_token = os.getenv("MD_AUTH_TOKEN")
assert api_token, "MD_AUTH_TOKEN not set (check your .env)"
base_url = os.getenv("MD_API_BASE_URL")

client = MDClient(api_token=api_token, base_url=base_url)
print("V2 client initialized.")

In [ ]:
# Print current md-python tag
import importlib.metadata
import json
import subprocess

direct_url_files = [p for p in importlib.metadata.files('md-python') if 'direct_url.json' in str(p)]
if direct_url_files:
    direct_url = json.loads(direct_url_files[0].read_text())
    url = direct_url.get('url', '')
    if 'tags/' in url:
        tag = url.rstrip('.tar.gz').split('/')[-1]
    elif url.startswith('file://'):
        repo_path = url[len('file://'):]
        result = subprocess.run(['git', 'describe', '--tags'], cwd=repo_path, capture_output=True, text=True)
        tag = result.stdout.strip() or '(no tag)'
    else:
        tag = '(unknown)'
    print(tag)
else:
    print('(no install metadata found)')

In [ ]:
health = client.health.check()
print("Health:", health)

In [ ]:
LOCAL_DATA_DIR = Path("/Users/giuseppeinfusini/wd/md-repos/cptac-pdac-md-converter/data/processed").resolve()
assert LOCAL_DATA_DIR.exists(), f"Path not found: {LOCAL_DATA_DIR}"

# Primary proteomics files to upload
# Add phospho_intensity.tsv or gene_expression.tsv here if needed
filenames = [
    "protein_intensity.tsv",
    "peptide_intensity_fake.tsv",
]

for f in filenames:
    assert (LOCAL_DATA_DIR / f).exists(), f"Missing: {f}"
print("Files to upload:", filenames)

In [ ]:
# Load design and metadata directly from CSVs
exp_design = ExperimentDesign.from_csv(str(LOCAL_DATA_DIR / "experiment_design.csv"))
sample_md = SampleMetadata.from_csv(str(LOCAL_DATA_DIR / "sample_metadata.csv"))

print(f"Design rows: {len(exp_design.data) - 1}")
print(f"Sample metadata rows: {len(sample_md.data) - 1}")
print(f"Sample metadata columns: {sample_md.data[0]}")

In [ ]:
exp_design

In [ ]:
sample_md

## Gene Expression Upload\n\nSeparate experiment for gene expression data using `md_format_gene` source. Same experiment design and sample metadata."

In [ ]:
gene_upload = Upload(
    name="CPTAC-PDAC Gene Expression - no expdesign",
    source="md_format_gene",
    # experiment_design=exp_design,
    sample_metadata=sample_md,
    file_location=str(LOCAL_DATA_DIR),
    filenames=["gene_expression.tsv"],
)

gene_upload_id = client.uploads.create(gene_upload)
print("Gene expression upload created. ID:", gene_upload_id)

In [ ]:
# Poll until gene expression upload reaches a terminal state
# gene_upload_id = "..."  # uncomment and set if resuming
completed_gene = client.uploads.wait_until_complete(gene_upload_id, poll_s=10, timeout_s=3600)
print(completed_gene)

In [ ]:
gene_initial_dataset = client.datasets.find_initial_dataset(gene_upload_id)
print("Gene expression initial dataset ID:", gene_initial_dataset.id)
gene_initial_dataset

In [ ]:
gene_pw = PairwiseComparisonDataset(
    input_dataset_ids=[str(gene_initial_dataset.id)],
    dataset_name="CPTAC-PDAC Gene Expression Pairwise Early vs Late",
    sample_metadata=sample_md,
    condition_column="condition",
    condition_comparisons=comparisons,
    entity_type="protein",
)
gene_pairwise_dataset_id = gene_pw.run(client)
print("Gene pairwise dataset ID:", gene_pairwise_dataset_id)

In [ ]:
gene_pw_state = client.datasets.wait_until_complete(
    upload_id=str(gene_upload_id),
    dataset_id=str(gene_pairwise_dataset_id),
    poll_s=10,
    timeout_s=3600,
)
print("Gene pairwise state:", gene_pw_state.state)
gene_pw_state